In [ ]:
import math

def std_pdf(x):
    return math.exp(-0.5 * x * x) / math.sqrt(2 * math.pi)

def std_cdf(x):
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def black_scholes_prices(S, K, T, r, sigma, q):
    if T <= 0:
        return max(0.0, S - K), max(0.0, K - S)

    sqrtT = math.sqrt(T)
    d1 = (math.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * sqrtT)
    d2 = d1 - sigma * sqrtT

    call = S * math.exp(-q * T) * std_cdf(d1) - K * math.exp(-r * T) * std_cdf(d2)
    put  = K * math.exp(-r * T) * std_cdf(-d2) - S * math.exp(-q * T) * std_cdf(-d1)

    return call, put

def black_scholes_greeks(S, K, T, r, sigma, q):
    if T <= 0:
        return {}

    sqrtT = math.sqrt(T)
    d1 = (math.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * sqrtT)
    d2 = d1 - sigma * sqrtT
    pdf_d1 = std_pdf(d1)

    delta_call = math.exp(-q * T) * std_cdf(d1)
    delta_put = math.exp(-q * T) * (std_cdf(d1) - 1)

    gamma = (math.exp(-q * T) * pdf_d1) / (S * sigma * sqrtT)

    # Adjust Vega: per 1% change
    vega = S * math.exp(-q * T) * pdf_d1 * sqrtT / 100

    # Adjust Theta: per day
    theta_call = ((-S * pdf_d1 * sigma * math.exp(-q * T) / (2 * sqrtT)
                   - r * K * math.exp(-r * T) * std_cdf(d2)
                   + q * S * math.exp(-q * T) * std_cdf(d1)) / 365)

    theta_put = ((-S * pdf_d1 * sigma * math.exp(-q * T) / (2 * sqrtT)
                  + r * K * math.exp(-r * T) * std_cdf(-d2)
                  - q * S * math.exp(-q * T) * std_cdf(-d1)) / 365)

    # Adjust Rho: per 1% change
    rho_call = (K * T * math.exp(-r * T) * std_cdf(d2)) / 100
    rho_put = (-K * T * math.exp(-r * T) * std_cdf(-d2)) / 100

    return {
        "Delta_call": delta_call,
        "Delta_put": delta_put,
        "Gamma": gamma,
        "Vega": vega,
        "Theta_call": theta_call,
        "Theta_put": theta_put,
        "Rho_call": rho_call,
        "Rho_put": rho_put
    }

if __name__ == "__main__":
    print("---- Black-Scholes-Merton Option Calculator ----")
    S = float(input("Enter Spot Price (S): "))
    K = float(input("Enter Strike Price (K): "))
    T = float(input("Enter Time to Maturity in Years (T): "))
    r = float(input("Enter Risk-Free Rate (r) in decimal (e.g., 0.06 for 6%): "))
    sigma = float(input("Enter Volatility (σ) in decimal (e.g., 0.25 for 25%): "))
    q = float(input("Enter Dividend Yield (q) in decimal (e.g., 0.02 for 2%): "))

    call, put = black_scholes_prices(S, K, T, r, sigma, q)
    greeks = black_scholes_greeks(S, K, T, r, sigma, q)

    print("\n--- Results ---")
    print(f"Call Option Price: {call:.4f}")
    print(f"Put Option Price:  {put:.4f}\n")

    for name, value in greeks.items():
        print(f"{name}: {value:.6f}")


---- Black-Scholes-Merton Option Calculator ----
Enter Spot Price (S): 100
Enter Strike Price (K): 150
Enter Time to Maturity in Years (T): 1
Enter Risk-Free Rate (r) in decimal (e.g., 0.06 for 6%): 0.07
Enter Volatility (σ) in decimal (e.g., 0.25 for 25%): 0.25
Enter Dividend Yield (q) in decimal (e.g., 0.02 for 2%): 0

--- Results ---
Call Option Price: 1.2239
Put Option Price:  41.0830

Delta_call: 0.111829
Delta_put: -0.888171
Gamma: 0.007611
Vega: 0.190270
Theta_call: -0.008426
Theta_put: 0.018396
Rho_call: 0.099589
Rho_put: -1.299001
